In [ ]:
import pandas as pd

df = pd.read_parquet("hf://datasets/ruslanmv/ai-medical-chatbot/dialogues.parquet")

In [ ]:

df.head()

In [ ]:
import re
import json

def clean_medical_text(text):
    if not isinstance(text, str): 
        return 
    text = re.sub(r'<.*?>', '', text)
    text = text.replace('-->', '').replace('==>', '')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_patient'] = df['Patient'].apply(clean_medical_text)
df['clean_description'] = df['Description'].apply(clean_medical_text)
df['clean_doctor'] = df['Doctor'].apply(clean_medical_text)

print("Data Cleaning Complete.")
df[['Patient', 'clean_patient']].head(10) 

In [ ]:
df['search_text'] = "SUMMARY: " + df['clean_description'] + " | PATIENT: " + df['clean_patient']

In [ ]:
import pandas as pd
import spacy
import spacy
import en_ner_bc5cdr_md
from tqdm import tqdm 

print("Loading SciSpaCy model...")
try:
    nlp = en_ner_bc5cdr_md.load()
except OSError:
    raise ImportError("Model not found!")

def extract_medical_entities_batch(text_series, batch_size=256):
    results = []
    total_docs = len(text_series)
    print(f"Processing {total_docs} rows with SciSpaCy...")
    for doc in tqdm(nlp.pipe(text_series, batch_size=batch_size), total=total_docs):
        entities = {}
        for ent in doc.ents:
            label = ent.label_
            text = ent.text.lower() 
            if label not in entities:
                entities[label] = []
            if text not in entities[label]:
                entities[label].append(text)
        results.append(entities)
    return results

df['metadata'] = extract_medical_entities_batch(df['search_text'])

print("\n--- RESULTS SAMPLE ---")
print(df[['search_text', 'metadata']].head())

output_file = "medical_data_with_metadata.json"
df.to_json(output_file, orient='records', indent=4)
print(f"\nSaved enriched dataset to: {output_file}")

In [ ]:
df.head()

In [ ]:
import pandas as pd
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm 
import time
import os
import torch
from dotenv import load_dotenv

load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY not found! Check your .env file.")

INDEX_NAME = "medical-bot"

COL_TEXT_TO_EMBED = "search_text" 
COL_RESPONSE = "Doctor"           
COL_METADATA = "metadata"         

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device Selected: {device.upper()}")

print("Connecting to Pinecone...")
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)

print(f"Loading AI Model on {device.upper()}...")
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

print(f"Starting Upload of {len(df)} rows...")

BATCH_SIZE = 200 

for i in tqdm(range(0, len(df), BATCH_SIZE), desc="Uploading Batches"):
    batch = df.iloc[i : i + BATCH_SIZE]
    
    try:
        embeddings = model.encode(batch[COL_TEXT_TO_EMBED].tolist(), batch_size=BATCH_SIZE).tolist()
    except Exception as e:
        print(f"Error encoding batch {i}: {e}")
        continue

    vectors_to_upload = []
    
    for j, row in enumerate(batch.itertuples()):
        response_text = getattr(row, COL_RESPONSE, "")
        if pd.isna(response_text): response_text = "No advice available."
        
        search_text = getattr(row, COL_TEXT_TO_EMBED, "")
        meta_tags = getattr(row, COL_METADATA, "{}")
        
        vectors_to_upload.append({
            "id": str(row.Index), 
            "values": embeddings[j],
            "metadata": {
                "text": str(search_text)[:1000],
                "response": str(response_text)[:2000],
                "tags": str(meta_tags)
            }
        })
    
    try:
        index.upsert(vectors=vectors_to_upload)
    except Exception as e:
        print(f"Error uploading batch {i}: {e}")
        time.sleep(1)

print("Upload Complete!")